# Embedding Similarity Demo

This notebook demonstrates how to load an embedding model and compute cosine similarity between a query and text examples.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

In [32]:
# Configuration
# MODEL_NAME = 'ibm-granite/granite-embedding-30m-english'
granite47m = 'ibm-granite/granite-embedding-small-english-r2'
granite149m = "ibm-granite/granite-embedding-english-r2"
qwen600m = 'Qwen/Qwen3-Embedding-0.6B'
gemma300m = 'google/embeddinggemma-300m'
granite30m = "ibm-granite/granite-embedding-30m-english"

In [7]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


In [31]:
from sentence_transformers import SentenceTransformer

# Load model using SentenceTransformer
MODEL_NAME = granite149m
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f"Loaded model: {MODEL_NAME}")


Loaded model: ibm-granite/granite-embedding-english-r2


# Example texts and query
documents = [
    "Python is a popular programming language known for its simple syntax.",
    "Machine learning models can be trained to recognize patterns in data.",
    "The Eiffel Tower is located in Paris, France.",
    "Neural networks are inspired by the structure of the human brain.",
    "Coffee is one of the most consumed beverages in the world.",
]

query = "What programming languages are easy to learn?"

In [9]:
def get_embeddings(texts, model, device):
    """Compute embeddings for a list of texts using SentenceTransformer."""
    # SentenceTransformer.encode handles tokenization, pooling, and normalization automatically
    embeddings = model.encode(texts, convert_to_tensor=True, device=device)
    return embeddings


In [24]:
# Example texts and query
documents = [
    "Minority interest In accounting, minority interest (or non-controlling interest) is the portion of a subsidiary corporation's stock that is not owned by the parent corporation. The magnitude of the minority interest in the subsidiary company is generally less than 50% of outstanding shares, or the corporation would generally cease to be a subsidiary of the parent.[1]",
    "In accounting, minority interest (or non-controlling interest) is the portion of a subsidiary corporation's stock that is not owned by the parent corporation. The magnitude of the minority interest in the subsidiary company is generally less than 50% of outstanding shares, or the corporation would generally cease to be a subsidiary of the parent.[1]",
    "Minority interest In accounting, non-controlling interest is the portion of a subsidiary corporation's stock that is not owned by the parent corporation. The magnitude of the minority interest in the subsidiary company is generally less than 50% of outstanding shares, or the corporation would generally cease to be a subsidiary of the parent.[1]",
    "Minority interest It is, however, possible (such as through special voting rights) for a controlling interest requiring consolidation to be achieved without exceeding 50% ownership, depending on the accounting standards being employed. Minority interest belongs to other investors and is reported on the consolidated balance sheet of the owning company to reflect the claim on assets belonging to other, non-controlling shareholders. Also, minority interest is reported on the consolidated income statement as a share of profit belonging to minority shareholders.",
    "Balance sheet Non-current assets (Fixed assets)",
    # "Balance sheet Non-current assets (Fixed assets) Minority interest In accounting, non-controlling interest is the portion of a subsidiary corporation's stock that is not owned by the parent corporation. The magnitude of the minority interest in the subsidiary company is generally less than 50% of outstanding shares, or the corporation would generally cease to be a subsidiary of the parent.[1]"
]

queries = [
    "what is non controlling interest on balance sheet?",
    "what is non controlling interest?",
    "what is minority interest on balance sheet?"
]

In [11]:
# Compute embeddings
query_embedding = get_embeddings([queries[0]], model, DEVICE)
doc_embeddings = get_embeddings(documents, model, DEVICE)

print(f"Query embedding shape: {query_embedding.shape}")
print(f"Document embeddings shape: {doc_embeddings.shape}")

Query embedding shape: torch.Size([1, 384])
Document embeddings shape: torch.Size([4, 384])


In [11]:
# Compute cosine similarity
def compute_query(_query, _documents):
    _query_embedding = get_embeddings([_query], model, DEVICE)
    _doc_embeddings = get_embeddings(_documents, model, DEVICE)

    similarities = F.cosine_similarity(_query_embedding, _doc_embeddings)

    print(f"Query: {_query}\n")
    print("Cosine similarities:")
    print("-" * 80)

    # Sort by similarity
    sorted_indices = similarities.argsort(descending=True)
    for idx in sorted_indices:
        print(f"Score: {similarities[idx]:.4f} | {_documents[idx]}")
    print("=" * 80)
    print()


In [25]:
for q in queries:
    compute_query(q, documents)

Query: what is non controlling interest on balance sheet?

Cosine similarities:
--------------------------------------------------------------------------------
Score: 0.7321 | Minority interest In accounting, non-controlling interest is the portion of a subsidiary corporation's stock that is not owned by the parent corporation. The magnitude of the minority interest in the subsidiary company is generally less than 50% of outstanding shares, or the corporation would generally cease to be a subsidiary of the parent.[1]
Score: 0.7019 | In accounting, minority interest (or non-controlling interest) is the portion of a subsidiary corporation's stock that is not owned by the parent corporation. The magnitude of the minority interest in the subsidiary company is generally less than 50% of outstanding shares, or the corporation would generally cease to be a subsidiary of the parent.[1]
Score: 0.6869 | Minority interest In accounting, minority interest (or non-controlling interest) is the port